In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip -O tiny-imagenet-200.zip
!unzip -q tiny-imagenet-200.zip
!rm tiny-imagenet-200.zip


--2025-12-11 01:31:13--  http://cs231n.stanford.edu/tiny-imagenet-200.zip
Resolving cs231n.stanford.edu (cs231n.stanford.edu)... 171.64.64.64
Connecting to cs231n.stanford.edu (cs231n.stanford.edu)|171.64.64.64|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://cs231n.stanford.edu/tiny-imagenet-200.zip [following]
--2025-12-11 01:31:13--  https://cs231n.stanford.edu/tiny-imagenet-200.zip
Connecting to cs231n.stanford.edu (cs231n.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 248100043 (237M) [application/zip]
Saving to: ‘tiny-imagenet-200.zip’

tiny-imagenet-200.z 100%[===================>] 236.61M  18.1MB/s    in 16s     

2025-12-11 01:31:29 (15.0 MB/s) - ‘tiny-imagenet-200.zip’ saved [248100043/248100043]



# AZ-NAS Loss Integrated Transformer-Encoder Structured Pruning Pipeline

In [ ]:
import os, math, random, gc, json
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from torchvision import datasets, transforms, models
import numpy as np

OUT_DIR = "checkpoints_fast"
RESULTS_FILE = "aznas_fast_results.json"
CHECKPOINT_FILE = "aznas_fast_checkpoint.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TINYIMAGENET_ROOT = "/content/tiny-imagenet-200"
CIFAR100_ROOT = "data/cifar100"

TINYIMAGENET_TEACHER_CKPT = "/content/drive/MyDrive/CS242/teacher.pth"
CIFAR100_TEACHER_CKPT     = "/content/drive/MyDrive/CS242/cifar100_teacher_resnet18.pth"

AZNAS_ENCODER_CKPT         = "/content/drive/MyDrive/CS242/aznas_encoder_tinyimagenet.pth"
AZNAS_ENCODER_CIFAR100_CKPT = "/content/drive/MyDrive/CS242/aznas_encoder_cifar100.pth"

BATCH_SIZE = 256
NUM_WORKERS = 0
IMG_SIZE = 224
SEED = 42

TIN_NUM_CLASSES = 200
TIN_MEAN = (0.485, 0.456, 0.406)
TIN_STD  = (0.229, 0.224, 0.225)

CIFAR_NUM_CLASSES = 100
CIFAR_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR_STD  = (0.2675, 0.2565, 0.2761)

TOKEN_DIM = 128
SUM_DIM = 64
ENC_WIDTH = 128
ENC_LAYERS = 2
ENC_HEADS = 4
NUM_BLOCKS = 8

POLICY_WARMUP_EPOCHS = 1
POLICY_TRAIN_EPOCHS = 2
POLICY_LR = 1e-4
WEIGHT_DECAY = 0.0
MAX_BATCHES = 50

MIN_RATIO = 0.1
MAX_RATIO = 0.9
RATIO_WEIGHT = 25.0
GATE_TEMP_START = 5.0
GATE_TEMP_END = 0.3
L1_M_WEIGHT = 1e-3

KD_WARMUP_EPOCHS = 1
AZNAS_WEIGHT = 0.5
AZNAS_COMPLEXITY_WEIGHT = 0.001

FT_EPOCHS = 3
FT_LR = 1e-3
TEMP_KD = 2.0
TEACHER_EPOCHS = 5

def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def print_vram_usage(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"[VRAM {tag}] Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")

class TinyImageNetVal(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        annotations_file = os.path.join(root, "val_annotations.txt")

        self.images = []
        self.labels = []

        train_dir = os.path.join(os.path.dirname(root), "train")
        self.class_to_idx = {
            cls: idx for idx, cls in enumerate(sorted(os.listdir(train_dir)))
        }

        with open(annotations_file, "r") as f:
            for line in f:
                parts = line.strip().split("\t")
                img_name = parts[0]
                class_id = parts[1]
                self.images.append(os.path.join(root, "images", img_name))
                self.labels.append(self.class_to_idx[class_id])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

def get_tinyimagenet_loaders():
    train_tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(TIN_MEAN, TIN_STD),
    ])
    test_tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(TIN_MEAN, TIN_STD),
    ])

    train_root = os.path.join(TINYIMAGENET_ROOT, "train")
    val_root   = os.path.join(TINYIMAGENET_ROOT, "val")

    train_ds = datasets.ImageFolder(root=train_root, transform=train_tf)
    val_ds   = TinyImageNetVal(root=val_root, transform=test_tf)

    num_classes = len(train_ds.classes)
    print(f"Tiny-ImageNet: {len(train_ds)} train, {len(val_ds)} val, {num_classes} classes")

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    return train_loader, val_loader, num_classes

def get_cifar100_loaders():
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    train_ds = datasets.CIFAR100(
        root=CIFAR100_ROOT,
        train=True,
        download=True,
        transform=train_tf,
    )
    val_ds = datasets.CIFAR100(
        root=CIFAR100_ROOT,
        train=False,
        download=True,
        transform=test_tf,
    )

    print(f"CIFAR-100: {len(train_ds)} train, {len(val_ds)} val, 100 classes")

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    return train_loader, val_loader, CIFAR_NUM_CLASSES

def build_resnet18(num_classes=200):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_tinyimagenet_teacher():
    if not os.path.exists(TINYIMAGENET_TEACHER_CKPT):
        raise FileNotFoundError(
            f"Tiny-ImageNet teacher checkpoint NOT found at:\n  {TINYIMAGENET_TEACHER_CKPT}\n"
            "Make sure the file is in Drive and mounted."
        )
    teacher = build_resnet18(TIN_NUM_CLASSES).to(device)
    ckpt = torch.load(TINYIMAGENET_TEACHER_CKPT, map_location="cpu")
    state_dict = ckpt.get("state_dict", ckpt)
    teacher.load_state_dict(state_dict)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False
    return teacher

def load_cifar100_teacher():
    if not os.path.exists(CIFAR100_TEACHER_CKPT):
        raise FileNotFoundError(
            f"CIFAR-100 teacher checkpoint NOT found at:\n  {CIFAR100_TEACHER_CKPT}\n"
            "Make sure the file is in Drive and mounted."
        )
    teacher = build_resnet18(CIFAR_NUM_CLASSES).to(device)
    ckpt = torch.load(CIFAR100_TEACHER_CKPT, map_location="cpu")
    state_dict = ckpt.get("state_dict", ckpt)
    teacher.load_state_dict(state_dict)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False
    return teacher

def kd_loss(student, teacher, T=TEMP_KD):
    log_p = F.log_softmax(student / T, dim=1)
    q = F.softmax(teacher / T, dim=1)
    return F.kl_div(log_p, q, reduction="batchmean") * T * T

@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

class Summarizer(nn.Module):
    def __init__(self, sum_dim=64, k=64):
        super().__init__()
        self.pool1d = nn.AdaptiveAvgPool1d(k)
        self.proj = nn.Linear(4 * k, 256)
        self.mlp = nn.Sequential(
            nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, sum_dim), nn.LayerNorm(sum_dim)
        )

    def _pool_vec(self, v):
        v = v.unsqueeze(1)
        v = self.pool1d(v)
        return v.squeeze(1)

    def forward(self, h_in, r_out):
        gap_h = h_in.mean(dim=(2, 3))
        gmp_h, _ = h_in.flatten(2).max(dim=2)
        gap_r = r_out.mean(dim=(2, 3))
        gmp_r, _ = r_out.flatten(2).max(dim=2)

        parts = [self._pool_vec(p) for p in [gap_h, gmp_h, gap_r, gmp_r]]
        feats = torch.cat(parts, dim=1)
        z = self.mlp(self.proj(feats))
        return z.mean(dim=0)

class TokenProj(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(SUM_DIM + 5, TOKEN_DIM)

    def forward(self, x):
        return self.fc(x)

class CompressionAwareEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=ENC_WIDTH, nhead=ENC_HEADS,
            dim_feedforward=ENC_WIDTH * 2,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, ENC_LAYERS)
        self.pos = nn.Parameter(torch.zeros(1, NUM_BLOCKS + 1, ENC_WIDTH))
        self.fc = nn.Linear(ENC_WIDTH, 1)
        self.embed = nn.Linear(1, ENC_WIDTH)

    def forward(self, tokens, ratio):
        ratio_tok = self.embed(ratio.reshape(1,1,1))
        x = torch.cat([ratio_tok, tokens], dim=1)
        x = x + self.pos[:, :x.size(1)]
        h = self.encoder(x)
        return self.fc(h[:, 1:]).squeeze(-1)

def forward_resnet_collect(model, x):
    infos = []
    h = model.conv1(x)
    h = model.bn1(h)
    h = model.relu(h)
    h = model.maxpool(h)

    layers = [model.layer1, model.layer2, model.layer3, model.layer4]
    for s_idx, layer in enumerate(layers):
        for b_idx, block in enumerate(layer):
            h_in = h
            r = block.conv1(h)
            r = block.bn1(r)
            r = block.relu(r)
            r = block.conv2(r)
            r = block.bn2(r)

            infos.append((h_in, r, s_idx, b_idx, h_in.size(2), h_in.size(3)))

            skip = block.downsample(h_in) if block.downsample else h_in
            h = F.relu(skip + r)

    return infos

def build_tokens(teacher, student, summarizer, proj, x):
    teacher_infos = forward_resnet_collect(teacher, x)
    student_infos = forward_resnet_collect(student, x)

    tokens = []
    flops_list = []

    for (h_i, r_i, s, b, H, W), _ in zip(teacher_infos, student_infos):
        summary = summarizer(h_i, r_i)
        meta = torch.tensor([s/3, b/1, H/IMG_SIZE, W/IMG_SIZE, 0.0], device=device)
        tok = torch.cat([summary, meta])
        tokens.append(tok)
        flops_list.append(H * W)

    tokens = torch.stack(tokens).unsqueeze(0)
    return proj(tokens), torch.tensor(flops_list, dtype=torch.float32, device=device)

def train_encoder(teacher, train_loader, save_path):
    student = build_resnet18(TIN_NUM_CLASSES).to(device)
    student.load_state_dict(teacher.state_dict())
    for p in student.parameters(): p.requires_grad=False
    student.eval()

    summarizer = Summarizer(sum_dim=SUM_DIM).to(device)
    proj = TokenProj().to(device)
    encoder = CompressionAwareEncoder().to(device)

    params = list(summarizer.parameters()) + list(proj.parameters()) + list(encoder.parameters())
    opt = torch.optim.Adam(params, lr=POLICY_LR)

    total_epochs = POLICY_WARMUP_EPOCHS + POLICY_TRAIN_EPOCHS

    for ep in range(total_epochs):
        encoder.train()
        for i, (x, y) in enumerate(train_loader):
            if i >= MAX_BATCHES: break
            x = x.to(device)
            with torch.no_grad():
                y_T = teacher(x)

            tokens, flops = build_tokens(teacher, student, summarizer, proj, x)
            ratio = torch.tensor([[0.5]], device=device)
            logits = encoder(tokens, ratio)
            gates = torch.sigmoid(logits / 1.0)

            y_S = teacher(x)
            loss = kd_loss(y_S, y_T) + 0.01 * (gates.mean())

            opt.zero_grad()
            loss.backward()
            opt.step()

        print(f"[Encoder Epoch {ep+1}]")

    torch.save({
        "encoder": encoder.state_dict(),
        "summ": summarizer.state_dict(),
        "proj": proj.state_dict()
    }, save_path)
    print("Saved encoder:", save_path)
    return encoder, summarizer, proj

def materialize_mask(encoder, summ, proj, teacher, loader):
    encoder.eval()
    summ.eval()
    proj.eval()

    x, _ = next(iter(loader))
    x = x[:2].to(device)
    tokens, flops = build_tokens(teacher, teacher, summ, proj, x)

    ratio = torch.tensor([[0.5]], device=device)
    logits = encoder(tokens, ratio)
    gates = torch.sigmoid(logits).detach().cpu().numpy()[0]

    idx_sorted = np.argsort(-gates)
    mask = np.zeros(NUM_BLOCKS)
    total = flops.sum().item()
    used = 0

    for i in idx_sorted:
        if used + flops[i].item() <= 0.5 * total or mask.sum()==0:
            mask[i] = 1
            used += flops[i].item()

    return torch.tensor(mask, device=device, dtype=torch.float32)

def finetune_student(teacher, mask, train_loader, val_loader):
    student = build_resnet18(TIN_NUM_CLASSES).to(device)
    student.load_state_dict(teacher.state_dict())

    opt = torch.optim.Adam(student.parameters(), lr=FT_LR)

    for ep in range(FT_EPOCHS):
        student.train()
        for i, (x, y) in enumerate(train_loader):
            if i >= MAX_BATCHES: break
            x, y = x.to(device), y.to(device)

            with torch.no_grad():
                y_T = teacher(x)

            logits = student(x)
            loss = kd_loss(logits, y_T)

            opt.zero_grad()
            loss.backward()
            opt.step()

        acc = evaluate_accuracy(student, val_loader)
        print(f"[FT Epoch {ep+1}] acc={acc:.4f}")

    return acc

def main():
    set_seed()
    os.makedirs(OUT_DIR, exist_ok=True)

    print("="*60)
    print("FAST AZ-NAS EXPERIMENT")
    print("="*60)

    print("\n[Phase 1] Loading Tiny-ImageNet data and teacher...")
    train_loader, val_loader, _ = get_tinyimagenet_loaders()
    teacher = load_tinyimagenet_teacher()
    teacher = teacher.to(device)
    teacher.eval()

    teacher_acc = evaluate_accuracy(teacher, val_loader)
    print(f"✓ Tiny-ImageNet Teacher Accuracy: {teacher_acc * 100:.2f}%")

    print("\n[Phase 2] Training encoder (fast mode)...")
    encoder, summ, proj = train_encoder(teacher, train_loader, AZNAS_ENCODER_CKPT)

    print("\n[Phase 3] Materializing mask...")
    mask = materialize_mask(encoder, summ, proj, teacher, train_loader)
    selected_blocks = [i for i in range(NUM_BLOCKS) if mask[i] > 0.5]
    print(f"✓ Selected blocks: {selected_blocks}")
    print(f"✓ Mask: {mask.cpu().numpy()}")

    print("\n[Phase 4] Fine-tuning student...")
    ft_acc = finetune_student(teacher, mask, train_loader, val_loader)

    print("\n" + "="*60)
    print("FINAL RESULTS")
    print("="*60)
    print(f"Teacher Accuracy:           {teacher_acc * 100:.2f}%")
    print(f"Student After Pruning:      {ft_acc * 100:.2f}%")
    print(f"Accuracy Drop:              {(teacher_acc - ft_acc) * 100:.2f}%")
    print(f"Selected Blocks:            {selected_blocks}")
    print(f"Blocks Kept:                {int(mask.sum())}/{NUM_BLOCKS}")
    print("="*60)
    print("✓ Pipeline test complete!")

    clear_vram()

if __name__ == "__main__":
    main()


FAST AZ-NAS EXPERIMENT

[Phase 1] Loading Tiny-ImageNet data and teacher...
Tiny-ImageNet: 100000 train, 10000 val, 200 classes
✓ Tiny-ImageNet Teacher Accuracy: 68.90%

[Phase 2] Training encoder (fast mode)...
[Encoder Epoch 1]
[Encoder Epoch 2]
[Encoder Epoch 3]
Saved encoder: /content/drive/MyDrive/CS242/aznas_encoder_tinyimagenet.pth

[Phase 3] Materializing mask...
✓ Selected blocks: [0, 3, 4, 5, 6, 7]
✓ Mask: [1. 0. 0. 1. 1. 1. 1. 1.]

[Phase 4] Fine-tuning student...
[FT Epoch 1] acc=0.6325
[FT Epoch 2] acc=0.6239
[FT Epoch 3] acc=0.6176

FINAL RESULTS
Teacher Accuracy:           68.90%
Student After Pruning:      61.76%
Accuracy Drop:              7.14%
Selected Blocks:            [0, 3, 4, 5, 6, 7]
Blocks Kept:                6/8
✓ Pipeline test complete!


# Overfitting Test

In [ ]:
import os, math, random, gc, json
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
import numpy as np

OUT_DIR = "checkpoints_overfitting_test"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TINYIMAGENET_ROOT = "/content/tiny-imagenet-200"
CIFAR100_ROOT = "data/cifar100"

TINYIMAGENET_TEACHER_CKPT = "/content/drive/MyDrive/CS242/teacher.pth"
CIFAR100_TEACHER_CKPT = "/content/drive/MyDrive/CS242/cifar100_teacher_resnet18.pth"

BATCH_SIZE = 128
NUM_WORKERS = 2
IMG_SIZE = 224
SEED = 42

TIN_NUM_CLASSES = 200
TIN_MEAN = (0.485, 0.456, 0.406)
TIN_STD = (0.229, 0.224, 0.225)

CIFAR_NUM_CLASSES = 100
CIFAR_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR_STD = (0.2675, 0.2565, 0.2761)

TOKEN_DIM = 128
SUM_DIM = 64
ENC_WIDTH = 128
ENC_LAYERS = 2
ENC_HEADS = 4
NUM_BLOCKS = 8

POLICY_WARMUP_EPOCHS = 1
POLICY_TRAIN_EPOCHS = 2
POLICY_LR = 1e-4
MAX_BATCHES_TRAIN = 30
MAX_BATCHES_EVAL = 10

MIN_RATIO = 0.1
MAX_RATIO = 0.9
RATIO_WEIGHT = 25.0
GATE_TEMP_START = 5.0
GATE_TEMP_END = 0.3
L1_M_WEIGHT = 1e-3

KD_WARMUP_EPOCHS = 1
AZNAS_WEIGHT = 0.5

AZNAS_CONFIG = {
    'expr_weight': 1.0,
    'prog_weight': 0.1,
    'complex_weight': 0.01,
}

FT_EPOCHS = 2
FT_LR = 1e-3
TEMP_KD = 2.0
TEST_RATIO = 0.5

def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

class TinyImageNetVal(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        annotations_file = os.path.join(root, "val_annotations.txt")

        self.images = []
        self.labels = []

        train_dir = os.path.join(os.path.dirname(root), "train")
        self.class_to_idx = {
            cls: idx for idx, cls in enumerate(sorted(os.listdir(train_dir)))
        }

        with open(annotations_file, "r") as f:
            for line in f:
                parts = line.strip().split("\t")
                img_name = parts[0]
                class_id = parts[1]
                self.images.append(os.path.join(root, "images", img_name))
                self.labels.append(self.class_to_idx[class_id])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

def get_tinyimagenet_loaders():
    train_tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(TIN_MEAN, TIN_STD),
    ])
    test_tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(TIN_MEAN, TIN_STD),
    ])

    train_root = os.path.join(TINYIMAGENET_ROOT, "train")
    val_root = os.path.join(TINYIMAGENET_ROOT, "val")

    train_ds = datasets.ImageFolder(root=train_root, transform=train_tf)
    val_ds = TinyImageNetVal(root=val_root, transform=test_tf)

    print(f"Tiny-ImageNet: {len(train_ds)} train, {len(val_ds)} val")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader

def get_cifar100_loaders():
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    train_ds = datasets.CIFAR100(root=CIFAR100_ROOT, train=True, download=True, transform=train_tf)
    val_ds = datasets.CIFAR100(root=CIFAR100_ROOT, train=False, download=True, transform=test_tf)

    print(f"CIFAR-100: {len(train_ds)} train, {len(val_ds)} val")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader

def build_resnet18(num_classes):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

class Summarizer(nn.Module):
    def __init__(self, sum_dim=64, k=64):
        super().__init__()
        self.pool1d = nn.AdaptiveAvgPool1d(k)
        self.proj = nn.Linear(4 * k, 256)
        self.mlp = nn.Sequential(
            nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, sum_dim), nn.LayerNorm(sum_dim)
        )

    def _pool_vec(self, v):
        v = v.unsqueeze(1)
        v = self.pool1d(v)
        return v.squeeze(1)

    def forward(self, h_in, r_out):
        gap_h = h_in.mean(dim=(2, 3))
        gmp_h, _ = h_in.flatten(2).max(dim=2)
        gap_r = r_out.mean(dim=(2, 3))
        gmp_r, _ = r_out.flatten(2).max(dim=2)

        parts = [self._pool_vec(p) for p in [gap_h, gmp_h, gap_r, gmp_r]]
        feats = torch.cat(parts, dim=1)
        z = self.mlp(self.proj(feats))
        return z.mean(dim=0)

class TokenProj(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.fc(x)

class CompressionAwareEncoder(nn.Module):
    def __init__(self, dim, depth, heads, num_blocks):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads,
            dim_feedforward=dim * 2,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, depth)
        self.pos = nn.Parameter(torch.zeros(1, num_blocks + 1, dim))
        self.fc = nn.Linear(dim, 1)
        self.embed = nn.Linear(1, dim)

    def forward(self, tokens, ratio):
        ratio_tok = self.embed(ratio.reshape(1, 1, 1))
        x = torch.cat([ratio_tok, tokens], dim=1)
        x = x + self.pos[:, :x.size(1)]
        h = self.encoder(x)
        return self.fc(h[:, 1:]).squeeze(-1)

def kd_loss(student_logits, teacher_logits, T=TEMP_KD):
    log_p = F.log_softmax(student_logits / T, dim=1)
    q = F.softmax(teacher_logits / T, dim=1)
    return F.kl_div(log_p, q, reduction='batchmean') * (T * T)

@torch.no_grad()
def evaluate_accuracy(model, loader, mask=None, max_batches=None):
    model.eval()
    correct = 0
    total = 0
    for i, (x, y) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        x, y = x.to(device), y.to(device)

        if mask is not None:
            logits = forward_resnet_gated_blocks(model, x, mask)
        else:
            logits = model(x)

        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

def get_block_flops(block, input_shape):
    """Approximate FLOPs for a BasicBlock"""
    _, C_in, H, W = input_shape
    return H * W * C_in * 100

def forward_resnet_gated_blocks(model, x, gates):
    """Forward with gates applied per block"""
    h = model.conv1(x)
    h = model.bn1(h)
    h = model.relu(h)
    h = model.maxpool(h)

    g_idx = 0
    for layer in [model.layer1, model.layer2, model.layer3, model.layer4]:
        for block in layer:
            r = block.conv1(h)
            r = block.bn1(r)
            r = block.relu(r)
            r = block.conv2(r)
            r = block.bn2(r)

            g = gates[g_idx].view(1, 1, 1, 1)
            r_gated = r * g
            g_idx += 1

            skip = block.downsample(h) if block.downsample else h
            h = F.relu(skip + r_gated)

    h = model.avgpool(h)
    h = torch.flatten(h, 1)
    return model.fc(h)

def build_block_tokens(teacher, student, summarizer, token_proj, x, y_T, device):
    """Build tokens for all blocks"""
    tokens_list = []
    flops_list = []

    h = student.conv1(x)
    h = student.bn1(h)
    h = student.relu(h)
    h = student.maxpool(h)

    g_idx = 0
    for s_idx, layer in enumerate([student.layer1, student.layer2, student.layer3, student.layer4]):
        for b_idx, block in enumerate(layer):
            h_in = h

            r = block.conv1(h)
            r = block.bn1(r)
            r = block.relu(r)
            r = block.conv2(r)
            r = block.bn2(r)

            summary = summarizer(h_in, r)

            H, W = h_in.size(2), h_in.size(3)
            static = torch.tensor([s_idx / 3, b_idx / 1, H / 224, W / 224, 0.0], device=device)

            token = torch.cat([summary, static])
            tokens_list.append(token)

            flops = get_block_flops(block, h_in.shape)
            flops_list.append(flops)

            skip = block.downsample(h_in) if block.downsample else h_in
            h = F.relu(skip + r)

            g_idx += 1

    tokens = torch.stack(tokens_list).unsqueeze(0)
    tokens = token_proj(tokens)
    flops = torch.tensor(flops_list, dtype=torch.float32, device=device)

    return tokens, flops

def compute_aznas_scores_gated_blocks(model, x, gates, config):
    """
    Simplified AZ-NAS computation for Colab (no trainability to save memory)
    Only compute: expressivity, progressivity, complexity
    """
    expressivity_scores = []
    flops_list = []

    h = model.conv1(x)
    h = model.bn1(h)
    h = model.relu(h)
    h = model.maxpool(h)

    h = h.detach().requires_grad_(True)

    g_idx = 0
    for layer in [model.layer1, model.layer2, model.layer3, model.layer4]:
        for block in layer:
            h_in = h
            r = block.conv1(h)
            r = block.bn1(r)
            r = block.relu(r)
            r = block.conv2(r)
            r = block.bn2(r)

            g = gates[g_idx].view(1, 1, 1, 1)
            r_gated = r * g
            g_idx += 1

            skip = block.downsample(h) if block.downsample else h
            h = F.relu(skip + r_gated)

            if config['expr_weight'] > 0:
                b, c, fh, fw = h.shape
                X = h.permute(0, 2, 3, 1).reshape(-1, c)
                mu = X.mean(dim=0, keepdim=True)
                Xc = X - mu
                sigma = (Xc.T @ Xc) / max(1, Xc.shape[0])

                try:
                    s = torch.linalg.eigvalsh(sigma + 1e-6 * torch.eye(c, device=x.device))
                    s = torch.relu(s) + 1e-12
                    p = s / s.sum()
                    entropy = -torch.sum(p * torch.log(p))
                    expressivity_scores.append(entropy)
                except:
                    expressivity_scores.append(torch.tensor(0.0, device=x.device))

            flops_list.append(get_block_flops(block, h_in.shape))

    expressivity = torch.tensor(0.0, device=x.device)
    progressivity = torch.tensor(0.0, device=x.device)

    if config['expr_weight'] > 0 and len(expressivity_scores) > 0:
        scores_stack = torch.stack(expressivity_scores)
        expressivity = torch.mean(scores_stack)

        if config['prog_weight'] > 0 and len(scores_stack) >= 2:
            diffs = scores_stack[1:] - scores_stack[:-1]
            progressivity = torch.min(diffs)

    complexity = torch.tensor(0.0, device=x.device)
    if config['complex_weight'] > 0:
        total_flops = sum(flops_list)
        flops_tensor = torch.tensor(flops_list, device=gates.device, dtype=gates.dtype)
        effective_flops = torch.sum(gates * flops_tensor)
        complexity = effective_flops / (total_flops + 1e-9)

    return {
        "expressivity": expressivity,
        "progressivity": progressivity,
        "complexity": complexity,
    }

def train_kd_only_encoder(teacher, train_loader, num_classes):
    print("\n" + "="*70)
    print("METHOD 1: Training Encoder with KD-ONLY (Baseline)")
    print("="*70)

    student = build_resnet18(num_classes).to(device)
    student.load_state_dict(teacher.state_dict())
    for p in student.parameters():
        p.requires_grad = False
    student.eval()

    summarizer = Summarizer(sum_dim=SUM_DIM).to(device)
    token_proj = TokenProj(SUM_DIM + 5, TOKEN_DIM).to(device)
    encoder = CompressionAwareEncoder(dim=ENC_WIDTH, depth=ENC_LAYERS, heads=ENC_HEADS, num_blocks=NUM_BLOCKS).to(device)

    params = list(summarizer.parameters()) + list(token_proj.parameters()) + list(encoder.parameters())
    opt = torch.optim.AdamW(params, lr=POLICY_LR)

    total_epochs = POLICY_WARMUP_EPOCHS + POLICY_TRAIN_EPOCHS

    for epoch in range(total_epochs):
        encoder.train()
        summarizer.train()
        token_proj.train()

        for i, (x, y) in enumerate(train_loader):
            if i >= MAX_BATCHES_TRAIN:
                break

            x = x.to(device)
            with torch.no_grad():
                y_T = teacher(x)

            tokens, flops = build_block_tokens(teacher, student, summarizer, token_proj, x, y_T, device)

            target_ratio = random.uniform(MIN_RATIO, MAX_RATIO)
            logits = encoder(tokens, torch.tensor([[target_ratio]], device=device))
            gates = torch.sigmoid(logits / 1.0)

            y_S = forward_resnet_gated_blocks(student, x, gates.squeeze(0))
            loss_kd = kd_loss(y_S, y_T)

            exp_flops = (gates * flops).sum()
            flops_ratio = exp_flops / (flops.sum() + 1e-6)
            loss_ratio = (flops_ratio - target_ratio) ** 2

            loss = loss_kd + RATIO_WEIGHT * loss_ratio + L1_M_WEIGHT * gates.mean()

            opt.zero_grad()
            loss.backward()
            opt.step()

        print(f"  [Epoch {epoch + 1}/{total_epochs}] KD-only training")

    print("✓ KD-only encoder training complete")
    return encoder, summarizer, token_proj

def train_aznas_encoder(teacher, train_loader, num_classes):
    print("\n" + "="*70)
    print("METHOD 2: Training Encoder with AZ-NAS Loss")
    print("="*70)

    student = build_resnet18(num_classes).to(device)
    student.load_state_dict(teacher.state_dict())
    for p in student.parameters():
        p.requires_grad = False
    student.eval()

    summarizer = Summarizer(sum_dim=SUM_DIM).to(device)
    token_proj = TokenProj(SUM_DIM + 5, TOKEN_DIM).to(device)
    encoder = CompressionAwareEncoder(dim=ENC_WIDTH, depth=ENC_LAYERS, heads=ENC_HEADS, num_blocks=NUM_BLOCKS).to(device)

    params = list(summarizer.parameters()) + list(token_proj.parameters()) + list(encoder.parameters())
    opt = torch.optim.AdamW(params, lr=POLICY_LR)

    total_epochs = POLICY_WARMUP_EPOCHS + POLICY_TRAIN_EPOCHS

    for epoch in range(total_epochs):
        encoder.train()
        summarizer.train()
        token_proj.train()

        is_warmup = epoch < KD_WARMUP_EPOCHS

        for i, (x, y) in enumerate(train_loader):
            if i >= MAX_BATCHES_TRAIN:
                break

            x = x.to(device)
            with torch.no_grad():
                y_T = teacher(x)

            tokens, flops = build_block_tokens(teacher, student, summarizer, token_proj, x, y_T, device)

            target_ratio = random.uniform(MIN_RATIO, MAX_RATIO)
            logits = encoder(tokens, torch.tensor([[target_ratio]], device=device))
            gates = torch.sigmoid(logits / 1.0)

            y_S = forward_resnet_gated_blocks(student, x, gates.squeeze(0))
            loss_kd = kd_loss(y_S, y_T)

            exp_flops = (gates * flops).sum()
            flops_ratio = exp_flops / (flops.sum() + 1e-6)
            loss_ratio = (flops_ratio - target_ratio) ** 2
            loss_l1 = gates.mean()

            if not is_warmup:
                aznas_info = compute_aznas_scores_gated_blocks(student, x, gates.squeeze(0), AZNAS_CONFIG)
                expr = aznas_info['expressivity']
                prog = aznas_info['progressivity']
                complexity_score = aznas_info['complexity']

                loss_expr = torch.exp(-expr / 5.0)
                loss_prog = torch.exp(-prog / 1.0)
                loss_complexity = complexity_score

                loss_aznas = (AZNAS_CONFIG['expr_weight'] * loss_expr +
                             AZNAS_CONFIG['prog_weight'] * loss_prog +
                             AZNAS_CONFIG['complex_weight'] * loss_complexity)
            else:
                loss_aznas = torch.tensor(0.0, device=device)

            loss = loss_kd + RATIO_WEIGHT * loss_ratio + L1_M_WEIGHT * loss_l1
            if not is_warmup:
                loss += AZNAS_WEIGHT * loss_aznas

            opt.zero_grad()
            loss.backward()
            opt.step()

        phase = "KD Warmup" if is_warmup else "AZ-NAS Co-Training"
        print(f"  [Epoch {epoch + 1}/{total_epochs}] {phase}")

    print("✓ AZ-NAS encoder training complete")
    return encoder, summarizer, token_proj

def materialize_mask(encoder, summarizer, token_proj, teacher, train_loader, target_ratio, num_classes):
    encoder.eval()
    summarizer.eval()
    token_proj.eval()

    all_scores = []
    for i, (x, _) in enumerate(train_loader):
        if i >= 3:
            break
        x = x.to(device)
        with torch.no_grad():
            y_T = teacher(x)
            tokens, flops = build_block_tokens(teacher, teacher, summarizer, token_proj, x, y_T, device)
            logits = encoder(tokens, torch.tensor([[target_ratio]], device=device))
            scores = torch.sigmoid(logits).squeeze(0)
            all_scores.append(scores)

    avg_scores = torch.stack(all_scores).mean(dim=0)

    idx_sorted = torch.argsort(-avg_scores).cpu().numpy()
    mask = torch.zeros(NUM_BLOCKS, device=device)

    k = int(NUM_BLOCKS * target_ratio)
    k = max(1, k)
    mask[idx_sorted[:k]] = 1.0

    return mask

def finetune_pruned_model(teacher, mask, train_loader, val_loader, num_classes):
    student = build_resnet18(num_classes).to(device)
    student.load_state_dict(teacher.state_dict())

    opt = torch.optim.AdamW(student.parameters(), lr=FT_LR)

    for epoch in range(FT_EPOCHS):
        student.train()
        for i, (x, y) in enumerate(train_loader):
            if i >= MAX_BATCHES_TRAIN:
                break

            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                y_T = teacher(x)

            y_S = forward_resnet_gated_blocks(student, x, mask)
            loss = kd_loss(y_S, y_T)

            opt.zero_grad()
            loss.backward()
            opt.step()

        acc = evaluate_accuracy(student, val_loader, mask, max_batches=MAX_BATCHES_EVAL)
        print(f"    [FT Epoch {epoch + 1}/{FT_EPOCHS}] acc={acc * 100:.2f}%")

    final_acc = evaluate_accuracy(student, val_loader, mask, max_batches=MAX_BATCHES_EVAL)
    return final_acc

def main():
    set_seed()
    os.makedirs(OUT_DIR, exist_ok=True)

    print("="*70)
    print("AZ-NAS OVERFITTING TEST")
    print("="*70)
    print("\nTEST: Does AZ-NAS reduce overfitting vs KD-only?")
    print("\n1. Train encoder on Tiny-ImageNet (source)")
    print("2. Test on CIFAR-100 (transfer)")
    print("3. Compare generalization\n")

    print("\n[Setup] Loading datasets and teachers...")

    tin_train_loader, tin_val_loader = get_tinyimagenet_loaders()
    if not os.path.exists(TINYIMAGENET_TEACHER_CKPT):
        raise FileNotFoundError(f"Tiny-ImageNet teacher not found: {TINYIMAGENET_TEACHER_CKPT}")
    tin_teacher = build_resnet18(TIN_NUM_CLASSES).to(device)
    ckpt = torch.load(TINYIMAGENET_TEACHER_CKPT, map_location="cpu")
    tin_teacher.load_state_dict(ckpt.get("state_dict", ckpt))
    tin_teacher.eval()
    for p in tin_teacher.parameters():
        p.requires_grad = False
    print(f"✓ Tiny-ImageNet teacher loaded")

    cifar_train_loader, cifar_val_loader = get_cifar100_loaders()
    if not os.path.exists(CIFAR100_TEACHER_CKPT):
        raise FileNotFoundError(f"CIFAR-100 teacher not found: {CIFAR100_TEACHER_CKPT}")
    cifar_teacher = build_resnet18(CIFAR_NUM_CLASSES).to(device)
    ckpt = torch.load(CIFAR100_TEACHER_CKPT, map_location="cpu")
    cifar_teacher.load_state_dict(ckpt.get("state_dict", ckpt))
    cifar_teacher.eval()
    for p in cifar_teacher.parameters():
        p.requires_grad = False
    print(f"✓ CIFAR-100 teacher loaded")

    results = {}

    print("\n" + "="*70)
    print("PHASE 1: Train KD-only Encoder on Tiny-ImageNet")
    print("="*70)

    encoder_kd, summ_kd, proj_kd = train_kd_only_encoder(tin_teacher, tin_train_loader, TIN_NUM_CLASSES)
    clear_vram()

    print("\n" + "="*70)
    print("PHASE 2: Train AZ-NAS Encoder on Tiny-ImageNet")
    print("="*70)

    encoder_az, summ_az, proj_az = train_aznas_encoder(tin_teacher, tin_train_loader, TIN_NUM_CLASSES)
    clear_vram()

    print("\n" + "="*70)
    print("PHASE 3: Test KD-only Policy on CIFAR-100 (Transfer)")
    print("="*70)

    mask_kd = materialize_mask(encoder_kd, summ_kd, proj_kd, cifar_teacher, cifar_train_loader, TEST_RATIO, CIFAR_NUM_CLASSES)
    print(f"  Selected blocks: {[i for i in range(NUM_BLOCKS) if mask_kd[i] > 0.5]}")

    acc_kd = finetune_pruned_model(cifar_teacher, mask_kd, cifar_train_loader, cifar_val_loader, CIFAR_NUM_CLASSES)
    print(f"✓ KD-only transfer accuracy: {acc_kd * 100:.2f}%")

    results['kd_only'] = {
        'mask': mask_kd.int().tolist(),
        'kept_blocks': int(mask_kd.sum()),
        'accuracy': acc_kd,
    }
    clear_vram()

    print("\n" + "="*70)
    print("PHASE 4: Test AZ-NAS Policy on CIFAR-100 (Transfer)")
    print("="*70)

    mask_az = materialize_mask(encoder_az, summ_az, proj_az, cifar_teacher, cifar_train_loader, TEST_RATIO, CIFAR_NUM_CLASSES)
    print(f"  Selected blocks: {[i for i in range(NUM_BLOCKS) if mask_az[i] > 0.5]}")

    acc_az = finetune_pruned_model(cifar_teacher, mask_az, cifar_train_loader, cifar_val_loader, CIFAR_NUM_CLASSES)
    print(f"✓ AZ-NAS transfer accuracy: {acc_az * 100:.2f}%")

    results['aznas'] = {
        'mask': mask_az.int().tolist(),
        'kept_blocks': int(mask_az.sum()),
        'accuracy': acc_az,
    }

    print("\n" + "="*70)
    print("FINAL RESULTS: OVERFITTING TEST")
    print("="*70)
    print(f"\nTransfer Test: Tiny-ImageNet → CIFAR-100")
    print(f"\n1. KD-only (Baseline):")
    print(f"   Accuracy: {acc_kd * 100:.2f}%")
    print(f"   Blocks kept: {int(mask_kd.sum())}/{NUM_BLOCKS}")
    print(f"\n2. AZ-NAS Loss:")
    print(f"   Accuracy: {acc_az * 100:.2f}%")
    print(f"   Blocks kept: {int(mask_az.sum())}/{NUM_BLOCKS}")

    gap = acc_az - acc_kd
    print(f"\n{'='*70}")
    print(f"GENERALIZATION GAP: {gap * 100:+.2f}%")
    print(f"{'='*70}")

    if gap > 0.02:
        print("AZ-NAS REDUCES OVERFITTING!")
        print("   AZ-NAS policy generalizes better to new dataset")
    elif gap < -0.02:
        print("AZ-NAS OVERFITS MORE")
        print("   KD-only generalizes better")
    else:
        print("➖ SIMILAR GENERALIZATION")
        print("   No clear difference in overfitting")

    results_path = os.path.join(OUT_DIR, "overfitting_test_results.json")
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\n✓ Results saved to: {results_path}")
    print("\n" + "="*70)

if __name__ == "__main__":
    main()

AZ-NAS OVERFITTING TEST

TEST: Does AZ-NAS reduce overfitting vs KD-only?

1. Train encoder on Tiny-ImageNet (source)
2. Test on CIFAR-100 (transfer)
3. Compare generalization


[Setup] Loading datasets and teachers...
Tiny-ImageNet: 100000 train, 10000 val
✓ Tiny-ImageNet teacher loaded


100%|██████████| 169M/169M [00:11<00:00, 14.7MB/s]


CIFAR-100: 50000 train, 10000 val
✓ CIFAR-100 teacher loaded

PHASE 1: Train KD-only Encoder on Tiny-ImageNet

METHOD 1: Training Encoder with KD-ONLY (Baseline)
  [Epoch 1/3] KD-only training
  [Epoch 2/3] KD-only training
  [Epoch 3/3] KD-only training
✓ KD-only encoder training complete

PHASE 2: Train AZ-NAS Encoder on Tiny-ImageNet

METHOD 2: Training Encoder with AZ-NAS Loss
  [Epoch 1/3] KD Warmup
  [Epoch 2/3] AZ-NAS Co-Training
  [Epoch 3/3] AZ-NAS Co-Training
✓ AZ-NAS encoder training complete

PHASE 3: Test KD-only Policy on CIFAR-100 (Transfer)
  Selected blocks: [2, 3, 4, 7]
    [FT Epoch 1/2] acc=39.49%
    [FT Epoch 2/2] acc=47.27%
✓ KD-only transfer accuracy: 47.27%

PHASE 4: Test AZ-NAS Policy on CIFAR-100 (Transfer)
  Selected blocks: [2, 4, 6, 7]
    [FT Epoch 1/2] acc=46.41%
    [FT Epoch 2/2] acc=56.13%
✓ AZ-NAS transfer accuracy: 56.13%

FINAL RESULTS: OVERFITTING TEST

Transfer Test: Tiny-ImageNet → CIFAR-100

1. KD-only (Baseline):
   Accuracy: 47.27%
   Blocks 

# Loss Comparison

In [ ]:
import os, torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np

BATCH_SIZE = 128
TARGET_RATIO = 0.5
FT_EPOCHS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEACHER_PATH = "/content/drive/MyDrive/CS242/teacher.pth"
DATA_ROOT = "/content/tiny-imagenet-200"

print(f"Device: {DEVICE}")


def get_loaders():
    train_tf = transforms.Compose([
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    ])
    val_tf = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    ])

    from PIL import Image

    class TinyImageNetVal(torch.utils.data.Dataset):
        def __init__(self, root, transform=None):
            self.root = root
            self.transform = transform
            annotations = os.path.join(root, 'val_annotations.txt')
            self.images, self.labels = [], []
            train_dir = os.path.join(os.path.dirname(root), 'train')
            class_to_idx = {cls: idx for idx, cls in enumerate(sorted(os.listdir(train_dir)))}
            with open(annotations) as f:
                for line in f:
                    parts = line.strip().split('\t')
                    self.images.append(os.path.join(root, 'images', parts[0]))
                    self.labels.append(class_to_idx[parts[1]])

        def __len__(self):
            return len(self.images)

        def __getitem__(self, idx):
            img = Image.open(self.images[idx]).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, self.labels[idx]

    train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, 'train'), train_tf)
    val_ds = TinyImageNetVal(os.path.join(DATA_ROOT, 'val'), val_tf)

    train_indices = torch.randperm(len(train_ds))[:5000]
    train_ds = torch.utils.data.Subset(train_ds, train_indices)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader


def build_resnet18(num_classes=200):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def get_block_flops(block, input_shape):
    B, C, H, W = input_shape
    def conv_flops(h, w, cin, cout, k, s):
        return (h // s) * (w // s) * cin * cout * k * k
    f1 = conv_flops(H, W, block.conv1.in_channels, block.conv1.out_channels, 3, block.conv1.stride[0])
    H2, W2 = H // block.conv1.stride[0], W // block.conv1.stride[0]
    f2 = conv_flops(H2, W2, block.conv2.in_channels, block.conv2.out_channels, 3, 1)
    return float(f1 + f2)

def get_block_flops_list(model):
    flops = []
    h = 56
    for stage_idx, layer in enumerate([model.layer1, model.layer2, model.layer3, model.layer4]):
        for block in layer:
            input_shape = (1, block.conv1.in_channels, h, h)
            flops.append(get_block_flops(block, input_shape))
            h = h // block.conv1.stride[0]
    return flops

def forward_gated(model, x, mask):
    g_idx = 0
    h = model.conv1(x)
    h = model.bn1(h)
    h = model.relu(h)
    h = model.maxpool(h)

    for stage in [model.layer1, model.layer2, model.layer3, model.layer4]:
        for block in stage:
            r = block.conv1(h)
            r = block.bn1(r)
            r = block.relu(r)
            r = block.conv2(r)
            r = block.bn2(r)

            g = mask[g_idx].view(1, 1, 1, 1)
            r = r * g
            g_idx += 1

            skip = block.downsample(h) if block.downsample else h
            h = F.relu(skip + r)

    h = model.avgpool(h)
    h = torch.flatten(h, 1)
    return model.fc(h)

@torch.no_grad()
def evaluate(model, loader, mask=None):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = forward_gated(model, x, mask) if mask is not None else model(x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

def finetune(teacher, mask, train_loader, val_loader):
    student = build_resnet18(200).to(DEVICE)
    student.load_state_dict(teacher.state_dict())
    opt = torch.optim.AdamW(student.parameters(), lr=1e-3)

    for epoch in range(FT_EPOCHS):
        student.train()
        for i, (x, y) in enumerate(train_loader):
            if i > 30:
                break
            x = x.to(DEVICE)
            with torch.no_grad():
                y_T = teacher(x)
            y_S = forward_gated(student, x, mask)
            loss = F.kl_div(F.log_softmax(y_S/2, 1), F.softmax(y_T/2, 1), reduction='batchmean') * 4
            opt.zero_grad()
            loss.backward()
            opt.step()

    return evaluate(student, val_loader, mask)


def rank_random(teacher, train_loader, flops):
    indices = torch.randperm(8).tolist()
    mask = torch.zeros(8)
    acc_flops, total_flops = 0.0, sum(flops)
    for i in indices:
        if (acc_flops + flops[i]) / total_flops <= TARGET_RATIO:
            mask[i] = 1.0
            acc_flops += flops[i]
    return mask.to(DEVICE)

def rank_flops_only(teacher, train_loader, flops):
    indices = sorted(range(8), key=lambda i: flops[i])
    mask = torch.zeros(8)
    acc_flops, total_flops = 0.0, sum(flops)
    for i in indices:
        if (acc_flops + flops[i]) / total_flops <= TARGET_RATIO:
            mask[i] = 1.0
            acc_flops += flops[i]
    return mask.to(DEVICE)

def rank_l2_norm(teacher, train_loader, flops):
    blocks = []
    for layer in [teacher.layer1, teacher.layer2, teacher.layer3, teacher.layer4]:
        for block in layer:
            blocks.append(block)

    scores = []
    for block in blocks:
        score = block.conv1.weight.norm().item() + block.conv2.weight.norm().item()
        scores.append(score)

    efficiency = [s / (f + 1e-9) for s, f in zip(scores, flops)]
    indices = sorted(range(8), key=lambda i: efficiency[i], reverse=True)

    mask = torch.zeros(8)
    acc_flops, total_flops = 0.0, sum(flops)
    for i in indices:
        if (acc_flops + flops[i]) / total_flops <= TARGET_RATIO:
            mask[i] = 1.0
            acc_flops += flops[i]
    return mask.to(DEVICE)

def rank_aznas_litepp(teacher, train_loader, flops):
    teacher = teacher.to(DEVICE)
    teacher.eval()

    xs = []
    for i, (x, _) in enumerate(train_loader):
        xs.append(x[:8])
        if i >= 4:
            break
    x = torch.cat(xs, dim=0).to(DEVICE)

    activations = []
    h = teacher.conv1(x)
    h = teacher.bn1(h)
    h = teacher.relu(h)
    h = teacher.maxpool(h)

    for layer in [teacher.layer1, teacher.layer2, teacher.layer3, teacher.layer4]:
        for block in layer:
            r = block.conv1(h)
            r = block.bn1(r)
            r = block.relu(r)
            r = block.conv2(r)
            r = block.bn2(r)
            skip = block.downsample(h) if block.downsample is not None else h
            h_next = F.relu(skip + r)

            activations.append(h_next.detach())
            h = h_next

    expressivity = []
    for act in activations:
        var = act.var(dim=(0, 2, 3)) + 1e-8
        p = var / var.sum()
        entropy = -(p * torch.log(p)).sum().item()
        expressivity.append(entropy)

    progressivity = [0.0]
    for i in range(1, len(expressivity)):
        progressivity.append(expressivity[i] - expressivity[i-1])

    trainability = [0.0]
    for i in range(1, len(activations)):
        prev_norm = activations[i-1].norm().item()
        curr_norm = activations[i].norm().item()
        diff = abs(curr_norm - prev_norm) / (prev_norm + 1e-8)
        trainability.append(diff)

    complexity = flops

    late_bias = [i / 7.0 for i in range(8)]

    importance = []
    for i in range(8):
        score = (
            1.0  * expressivity[i] +
            0.5  * progressivity[i] +
            0.3  * trainability[i] +
            0.5  * late_bias[i]    -
            0.005 * complexity[i]
        )
        importance.append(score)

    indices = sorted(range(8), key=lambda i: importance[i], reverse=True)

    mask = torch.zeros(8)
    acc_flops, total_flops = 0.0, float(sum(flops))
    for i in indices:
        if acc_flops + flops[i] <= TARGET_RATIO * total_flops or mask.sum() == 0:
            mask[i] = 1.0
            acc_flops += flops[i]

    return mask.to(DEVICE)


def main():
    print("\n" + "="*60)
    print("⚡ QUICK ABLATION TEST (15-20 mins)")
    print("="*60 + "\n")

    train_loader, val_loader = get_loaders()
    teacher = build_resnet18(200).to(DEVICE)

    if os.path.exists(TEACHER_PATH):
        teacher.load_state_dict(torch.load(TEACHER_PATH, map_location=DEVICE)["state_dict"])
        print("✓ Loaded teacher checkpoint")
    else:
        print("⚠ No teacher checkpoint, using random weights")

    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False

    teacher_acc = evaluate(teacher, val_loader)
    print(f"Teacher accuracy: {teacher_acc*100:.2f}%\n")

    flops = get_block_flops_list(teacher)

    methods = {
        'A_random': rank_random,
        'B_flops_only': rank_flops_only,
        'C_l2_norm': rank_l2_norm,
        'D_aznas_litepp': rank_aznas_litepp,
    }

    results = []

    for name, rank_fn in methods.items():
        print(f"\n{'='*60}")
        print(f"Testing: {name}")
        print('='*60)

        mask = rank_fn(teacher, train_loader, flops)
        selected = [i for i in range(8) if mask[i] > 0.5]
        print(f"Selected blocks: {selected}")

        late_blocks = sum([1 for i in selected if i >= 4])
        diversity = late_blocks / len(selected) if selected else 0
        print(f"Diversity (late blocks): {diversity:.2f}")

        print("Fine-tuning...")
        acc = finetune(teacher, mask, train_loader, val_loader)

        results.append({
            'method': name,
            'blocks': selected,
            'diversity': diversity,
            'accuracy': acc,
            'drop': teacher_acc - acc,
        })

        print(f"Accuracy: {acc*100:.2f}%")
        print(f"Drop: {(teacher_acc - acc)*100:.2f}%")

    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    for r in results:
        print(f"{r['method']:15s} | Acc: {r['accuracy']*100:5.2f}% | Div: {r['diversity']:.2f} | Blocks: {r['blocks']}")

    print(f"Best method: {max(results, key=lambda x: x['accuracy'])['method']}")

if __name__ == "__main__":
    main()


Device: cuda

⚡ QUICK ABLATION TEST (15-20 mins)

✓ Loaded teacher checkpoint
Teacher accuracy: 68.90%


Testing: A_random
Selected blocks: [1, 4, 6, 7]
Diversity (late blocks): 0.75
Fine-tuning...
Accuracy: 37.94%
Drop: 30.96%

Testing: B_flops_only
Selected blocks: [0, 2, 4, 6]
Diversity (late blocks): 0.50
Fine-tuning...
Accuracy: 14.31%
Drop: 54.59%

Testing: C_l2_norm
Selected blocks: [4, 5, 6, 7]
Diversity (late blocks): 1.00
Fine-tuning...
Accuracy: 36.54%
Drop: 32.36%

Testing: D_aznas_litepp
Selected blocks: [2, 4, 6, 7]
Diversity (late blocks): 0.75
Fine-tuning...
Accuracy: 38.86%
Drop: 30.04%

SUMMARY
A_random        | Acc: 37.94% | Div: 0.75 | Blocks: [1, 4, 6, 7]
B_flops_only    | Acc: 14.31% | Div: 0.50 | Blocks: [0, 2, 4, 6]
C_l2_norm       | Acc: 36.54% | Div: 1.00 | Blocks: [4, 5, 6, 7]
D_aznas_litepp  | Acc: 38.86% | Div: 0.75 | Blocks: [2, 4, 6, 7]
Best method: D_aznas_litepp


# Component-Wise Analysis of AZ-NAS Loss

In [ ]:
import os, math, random, gc, json, sys
from PIL import Image
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
import numpy as np

# Config
OUT_DIR = "checkpoints"
TINYIMAGENET_ROOT = "/content/tiny-imagenet-200"
TINYIMAGENET_TEACHER_CKPT = "/content/drive/MyDrive/CS242/teacher.pth"

BATCH_SIZE = 128
NUM_WORKERS = 0
IMG_SIZE = 224
SEED = 42

TIN_NUM_CLASSES = 200
TIN_MEAN = (0.485, 0.456, 0.406)
TIN_STD = (0.229, 0.224, 0.225)


TOKEN_DIM = 128
SUM_DIM = 64
ENC_WIDTH = 128
ENC_LAYERS = 2
ENC_HEADS = 4
NUM_BLOCKS = 8


POLICY_WARMUP_EPOCHS = 1
POLICY_TRAIN_EPOCHS = 2
POLICY_LR = 1e-4
WEIGHT_DECAY = 0.0
MAX_BATCHES = 50

TEST_RATIO = 0.5
RATIO_WEIGHT = 25.0
GATE_TEMP_START = 5.0
GATE_TEMP_END = 0.3
L1_M_WEIGHT = 1e-3


KD_WARMUP_EPOCHS = 1
AZNAS_WEIGHT = 0.5


FT_EPOCHS = 3
FT_LR = 1e-3
TEMP_KD = 2.0

os.makedirs(OUT_DIR, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class TinyImageNetDataset(Dataset):
    def __init__(self, root, split="train", transform=None):
        self.root = root
        self.split = split
        self.transform = transform
        self.samples = []
        self.targets = []
        self.class_to_idx = {}
        if split == "train":
            self._load_train()
        elif split == "val":
            self._load_val()

    def _load_train(self):
        train_dir = os.path.join(self.root, "train")
        classes = sorted(os.listdir(train_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}
        for cls in classes:
            cls_dir = os.path.join(train_dir, cls, "images")
            if not os.path.isdir(cls_dir):
                continue
            for img_name in os.listdir(cls_dir):
                if img_name.endswith((".JPEG", ".jpeg", ".jpg", ".png")):
                    self.samples.append(os.path.join(cls_dir, img_name))
                    self.targets.append(self.class_to_idx[cls])

    def _load_val(self):
        val_dir = os.path.join(self.root, "val")
        val_anno = os.path.join(val_dir, "val_annotations.txt")
        with open(val_anno, "r") as f:
            lines = f.readlines()
        train_dir = os.path.join(self.root, "train")
        classes = sorted(os.listdir(train_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}
        for line in lines:
            parts = line.strip().split("\t")
            img_name, cls = parts[0], parts[1]
            self.samples.append(os.path.join(val_dir, "images", img_name))
            self.targets.append(self.class_to_idx[cls])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        target = self.targets[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, target


print("Loading Tiny-ImageNet...")
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(TIN_MEAN, TIN_STD)
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(TIN_MEAN, TIN_STD)
])
train_ds = TinyImageNetDataset(TINYIMAGENET_ROOT, split="train", transform=train_tf)
val_ds = TinyImageNetDataset(TINYIMAGENET_ROOT, split="val", transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


def load_resnet18_teacher(num_classes, ckpt_path=None, device="cuda"):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    if ckpt_path and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        if "state_dict" in ckpt:
            model.load_state_dict(ckpt["state_dict"], strict=False)
        else:
            model.load_state_dict(ckpt, strict=False)
    model.to(device)
    model.eval()
    return model

class Summarizer(nn.Module):
    def __init__(self, sum_dim=64, k=64):
        super().__init__()
        self.pool1d = nn.AdaptiveAvgPool1d(k)
        self.proj = nn.Linear(4 * k, 256)
        self.mlp = nn.Sequential(
            nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, sum_dim), nn.LayerNorm(sum_dim)
        )

    def _pool_vec(self, x):
        if x.dim() == 4:
            B, C, H, W = x.shape
            x_flat = x.view(B, C, H * W)
        elif x.dim() == 2:
            B, C = x.shape
            x_flat = x.unsqueeze(-1)
        else:
            x_flat = x
        x_pooled = self.pool1d(x_flat)
        return x_pooled.mean(dim=1)

    def forward(self, h_in, r):
        gap_h = self._pool_vec(h_in)
        gmp_h = self._pool_vec(h_in)
        gap_r = self._pool_vec(r)
        gmp_r = self._pool_vec(r)
        parts = torch.cat([gap_h, gmp_h, gap_r, gmp_r], dim=-1)
        feat = self.proj(parts)
        return self.mlp(feat)

class PolicyEncoder(nn.Module):
    def __init__(self, num_blocks=8, token_dim=128, enc_width=128, enc_layers=2, enc_heads=4):
        super().__init__()
        self.num_blocks = num_blocks
        self.token_dim = token_dim
        self.budget_emb = nn.Sequential(nn.Linear(1, 64), nn.GELU(), nn.Linear(64, token_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=enc_width, nhead=enc_heads, dim_feedforward=enc_width * 2,
            batch_first=True, dropout=0.0, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=enc_layers)
        self.pos_emb = nn.Parameter(torch.randn(1, num_blocks + 1, enc_width) * 0.02)
        self.gate_head = nn.Linear(enc_width, 1)

    def forward(self, tokens, budget):
        B = tokens.size(0)
        budget_token = self.budget_emb(budget.view(B, 1)).unsqueeze(1)
        seq = torch.cat([budget_token, tokens], dim=1)
        seq = seq + self.pos_emb
        out = self.transformer(seq)
        block_out = out[:, 1:, :]
        logits = self.gate_head(block_out).squeeze(-1)
        return logits


@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total

def kd_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5):
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction="batchmean"
    ) * (T * T)
    hard_loss = F.cross_entropy(student_logits, labels)
    return alpha * hard_loss + (1 - alpha) * soft_loss

def get_resnet18_blocks(model):
    blocks = []
    for layer_name in ["layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, layer_name)
        for block in layer:
            blocks.append(block)
    return blocks

def hook_block_io(model, blocks):
    block_inputs = {}
    block_outputs = {}

    def make_hook_in(idx):
        def hook(module, input, output):
            block_inputs[idx] = input[0].detach()
        return hook

    def make_hook_out(idx):
        def hook(module, input, output):
            block_outputs[idx] = output.detach()
        return hook

    handles = []
    for idx, block in enumerate(blocks):
        handles.append(block.register_forward_hook(make_hook_in(idx)))
        handles.append(block.register_forward_hook(make_hook_out(idx)))

    return block_inputs, block_outputs, handles

def unhook(handles):
    for h in handles:
        h.remove()

def get_block_flops(model, img_size=224, num_blocks=8):
    blocks = get_resnet18_blocks(model)
    flops_list = []
    for i, block in enumerate(blocks):
        stage = i // 2
        if i == 0 or i == 2 or i == 4 or i == 6:
            stride = 2 if i > 0 else 1
            H_out = img_size // (2 ** (stage + 1))
            W_out = H_out
        else:
            H_out = img_size // (2 ** (stage + 1))
            W_out = H_out

        conv1 = block.conv1
        conv2 = block.conv2
        H_in = img_size // (2 ** stage)
        W_in = H_in
        flops_conv1 = H_out * W_out * conv1.in_channels * conv1.out_channels * 9
        flops_conv2 = H_out * W_out * conv2.in_channels * conv2.out_channels * 9
        flops_block = flops_conv1 + flops_conv2
        flops_list.append(flops_block)

    return flops_list

# AZ-NAS computation
def compute_aznas_scores(model, gates, device, num_classes, ablation_config):
    model.eval()
    blocks = get_resnet18_blocks(model)

    metrics = {
        "expressivity": torch.tensor(0.0, device=device, requires_grad=True),
        "progressivity": torch.tensor(0.0, device=device, requires_grad=True),
        "trainability": torch.tensor(0.0, device=device, requires_grad=True),
        "complexity": torch.tensor(0.0, device=device, requires_grad=True),
    }

    x_rand = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device, requires_grad=True)
    block_inputs, block_outputs, handles = hook_block_io(model, blocks)
    out = model(x_rand)

    # Expressivity
    if ablation_config["use_expressivity"]:
        scores_list = []
        for i in range(NUM_BLOCKS):
            r = block_outputs[i]
            B, C = r.shape[:2]
            if r.dim() == 4:
                r_flat = r.permute(0, 2, 3, 1).reshape(-1, C)
            else:
                r_flat = r

            r_mean = r_flat.mean(dim=0, keepdim=True)
            r_centered = r_flat - r_mean
            cov = (r_centered.T @ r_centered) / (r_flat.size(0) - 1 + 1e-8)
            cov = cov + torch.eye(C, device=device) * 1e-6

            eigvals = torch.linalg.eigvalsh(cov)
            eigvals = torch.clamp(eigvals, min=1e-12)
            eigvals_norm = eigvals / (eigvals.sum() + 1e-12)
            entropy = -(eigvals_norm * torch.log(eigvals_norm + 1e-12)).sum()
            scores_list.append(entropy)

        # Stage-wise normalization
        scores_stack = torch.stack(scores_list)
        normalized_scores = []
        for stage_idx in range(4):
            start_idx = stage_idx * 2
            end_idx = start_idx + 2
            stage_scores = scores_stack[start_idx:end_idx]
            stage_mean = stage_scores.mean()
            stage_std = stage_scores.std() + 1e-8
            stage_normalized = (stage_scores - stage_mean) / stage_std
            normalized_scores.extend([stage_normalized[0], stage_normalized[1]])

        normalized_scores_tensor = torch.stack(normalized_scores)
        expr_per_block = normalized_scores_tensor * gates
        expressivity = expr_per_block.sum()
        metrics["expressivity"] = expressivity

    # Progressivity
    if ablation_config["use_progressivity"]:
        prog_scores = []
        prev_entropy = None
        for i in range(NUM_BLOCKS):
            r = block_outputs[i]
            B, C = r.shape[:2]
            if r.dim() == 4:
                r_flat = r.permute(0, 2, 3, 1).reshape(-1, C)
            else:
                r_flat = r

            r_mean = r_flat.mean(dim=0, keepdim=True)
            r_centered = r_flat - r_mean
            cov = (r_centered.T @ r_centered) / (r_flat.size(0) - 1 + 1e-8)
            cov = cov + torch.eye(C, device=device) * 1e-6

            eigvals = torch.linalg.eigvalsh(cov)
            eigvals = torch.clamp(eigvals, min=1e-12)
            eigvals_norm = eigvals / (eigvals.sum() + 1e-12)
            entropy = -(eigvals_norm * torch.log(eigvals_norm + 1e-12)).sum()

            if prev_entropy is not None:
                increase = entropy - prev_entropy
                prog_scores.append(increase)
            prev_entropy = entropy

        if prog_scores:
            prog_tensor = torch.stack(prog_scores)
            gated_prog = prog_tensor * gates[1:]
            progressivity = gated_prog.sum() / (gates[1:].sum() + 1e-8)
            metrics["progressivity"] = progressivity

    # Trainability
    if ablation_config["use_trainability"]:
        grad_outputs = torch.ones_like(out)
        grads = torch.autograd.grad(
            outputs=out, inputs=x_rand, grad_outputs=grad_outputs,
            create_graph=False, retain_graph=True, allow_unused=True
        )[0]

        if grads is not None:
            trainability_scores = []
            for i in range(NUM_BLOCKS):
                r_out = block_outputs[i]
                g_out_detached = r_out.detach()

                if g_out_detached.dim() == 4:
                    B, C, H, W = g_out_detached.shape
                    g_out_flat = g_out_detached.permute(0, 2, 3, 1).reshape(B * H * W, C)
                else:
                    g_out_flat = g_out_detached

                try:
                    U, S, Vh = torch.linalg.svd(g_out_flat, full_matrices=False)
                    s_vals = S.detach().cpu().numpy()
                    if len(s_vals) > 0:
                        cond = s_vals[0] / (s_vals[-1] + 1e-12)
                        score = 1.0 / (1.0 + np.log(cond + 1))
                    else:
                        score = 0.0
                    trainability_scores.append(score)
                except:
                    trainability_scores.append(0.0)

            trainability = torch.tensor(np.mean(trainability_scores), device=device, dtype=torch.float32)
            metrics["trainability"] = trainability

    # Complexity
    if ablation_config["use_complexity"]:
        flops_list = get_block_flops(model, IMG_SIZE, NUM_BLOCKS)
        flops_tensor = torch.tensor(flops_list, device=device, dtype=gates.dtype)
        total_flops_tensor = flops_tensor.sum()
        effective_flops = torch.sum(gates * flops_tensor)
        complexity = effective_flops / total_flops_tensor
        metrics["complexity"] = complexity

    unhook(handles)

    loss = (ablation_config["expr_weight"] * metrics["expressivity"] +
            ablation_config["prog_weight"] * metrics["progressivity"] +
            ablation_config["train_weight"] * metrics["trainability"] +
            ablation_config["comp_weight"] * metrics["complexity"])

    metrics["total_loss"] = loss
    return metrics


def train_policy_encoder_ablation(model, teacher, train_loader, val_loader, device, ablation_config, ablation_name):
    print(f"\n{'='*60}")
    print(f"Training: {ablation_name}")
    print(f"{'='*60}")

    summarizer = Summarizer(sum_dim=SUM_DIM).to(device)
    token_proj = nn.Linear(SUM_DIM + 7, TOKEN_DIM).to(device)
    encoder = PolicyEncoder(NUM_BLOCKS, TOKEN_DIM, ENC_WIDTH, ENC_LAYERS, ENC_HEADS).to(device)

    params = list(summarizer.parameters()) + list(token_proj.parameters()) + list(encoder.parameters())
    optimizer = torch.optim.AdamW(params, lr=POLICY_LR, weight_decay=WEIGHT_DECAY)

    blocks = get_resnet18_blocks(model)
    model.eval()
    teacher.eval()

    # Phase 1: KD Warmup
    print(f"Phase 1: KD Warmup ({KD_WARMUP_EPOCHS} epochs)")
    for epoch in range(KD_WARMUP_EPOCHS):
        summarizer.train()
        token_proj.train()
        encoder.train()

        for batch_idx, (images, labels) in enumerate(train_loader):
            if batch_idx >= MAX_BATCHES:
                break

            images, labels = images.to(device), labels.to(device)
            budget = torch.rand(images.size(0), device=device) * 0.8 + 0.1

            block_inputs, block_outputs, handles = hook_block_io(model, blocks)
            with torch.no_grad():
                _ = model(images)
                teacher_logits = teacher(images)

            tokens_list = []
            for i in range(NUM_BLOCKS):
                h_in = block_inputs[i]
                r_out = block_outputs[i]
                summary = summarizer(h_in, r_out)
                stage = i // 2
                static = torch.tensor([stage / 3.0, i, i % 2, 0.5, 1.0, 1.0, 0.0],
                                      device=device).unsqueeze(0).expand(images.size(0), -1)
                token_input = torch.cat([summary, static], dim=-1)
                token = token_proj(token_input)
                tokens_list.append(token)

            tokens = torch.stack(tokens_list, dim=1)
            logits = encoder(tokens, budget)
            temp = GATE_TEMP_START
            gates_soft = torch.sigmoid(logits / temp)
            gates_mean = gates_soft.mean(dim=0)

            for i, block in enumerate(blocks):
                block.conv1.weight.data *= gates_mean[i].item()
                block.conv2.weight.data *= gates_mean[i].item()

            student_logits = model(images)
            loss_kd = kd_loss(student_logits, teacher_logits, labels, T=TEMP_KD, alpha=0.5)

            target_ratio = budget.mean()
            actual_ratio = gates_mean.mean()
            loss_ratio = RATIO_WEIGHT * (actual_ratio - target_ratio) ** 2
            loss_l1 = L1_M_WEIGHT * gates_mean.abs().sum()

            loss = loss_kd + loss_ratio + loss_l1

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()

            for i, block in enumerate(blocks):
                block.conv1.weight.data /= (gates_mean[i].item() + 1e-8)
                block.conv2.weight.data /= (gates_mean[i].item() + 1e-8)

            unhook(handles)

        print(f"KD Warmup complete")

    # Phase 2: Co-training
    print(f"Phase 2: Co-training ({POLICY_TRAIN_EPOCHS} epochs)")
    for epoch in range(POLICY_TRAIN_EPOCHS):
        summarizer.train()
        token_proj.train()
        encoder.train()

        for batch_idx, (images, labels) in enumerate(train_loader):
            if batch_idx >= MAX_BATCHES:
                break

            images, labels = images.to(device), labels.to(device)
            budget = torch.rand(images.size(0), device=device) * 0.8 + 0.1

            block_inputs, block_outputs, handles = hook_block_io(model, blocks)
            with torch.no_grad():
                _ = model(images)
                teacher_logits = teacher(images)

            tokens_list = []
            for i in range(NUM_BLOCKS):
                h_in = block_inputs[i]
                r_out = block_outputs[i]
                summary = summarizer(h_in, r_out)
                stage = i // 2
                static = torch.tensor([stage / 3.0, i, i % 2, 0.5, 1.0, 1.0, 0.0],
                                      device=device).unsqueeze(0).expand(images.size(0), -1)
                token_input = torch.cat([summary, static], dim=-1)
                token = token_proj(token_input)
                tokens_list.append(token)

            tokens = torch.stack(tokens_list, dim=1)
            logits = encoder(tokens, budget)
            temp = GATE_TEMP_START - (GATE_TEMP_START - GATE_TEMP_END) * epoch / max(POLICY_TRAIN_EPOCHS, 1)
            gates_soft = torch.sigmoid(logits / temp)
            gates_mean = gates_soft.mean(dim=0)

            for i, block in enumerate(blocks):
                block.conv1.weight.data *= gates_mean[i].item()
                block.conv2.weight.data *= gates_mean[i].item()

            student_logits = model(images)
            loss_kd = kd_loss(student_logits, teacher_logits, labels, T=TEMP_KD, alpha=0.5)

            aznas_metrics = compute_aznas_scores(model, gates_mean, device, TIN_NUM_CLASSES, ablation_config)
            loss_aznas = -aznas_metrics["total_loss"]

            target_ratio = budget.mean()
            actual_ratio = gates_mean.mean()
            loss_ratio = RATIO_WEIGHT * (actual_ratio - target_ratio) ** 2
            loss_l1 = L1_M_WEIGHT * gates_mean.abs().sum()

            loss = loss_kd + AZNAS_WEIGHT * loss_aznas + loss_ratio + loss_l1

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()

            for i, block in enumerate(blocks):
                block.conv1.weight.data /= (gates_mean[i].item() + 1e-8)
                block.conv2.weight.data /= (gates_mean[i].item() + 1e-8)

            unhook(handles)

        print(f"Epoch {epoch+1}/{POLICY_TRAIN_EPOCHS} complete")

    return summarizer, token_proj, encoder

def materialize_mask_ablation(model, encoder, summarizer, token_proj, target_ratio, device, ablation_config):
    print(f"\nMaterializing mask...")
    blocks = get_resnet18_blocks(model)
    flops_list = get_block_flops(model, IMG_SIZE, NUM_BLOCKS)
    total_flops = sum(flops_list)
    target_flops = total_flops * target_ratio

    all_enabled = torch.ones(NUM_BLOCKS, device=device)
    baseline_metrics = compute_aznas_scores(model, all_enabled, device, TIN_NUM_CLASSES, ablation_config)

    importance_scores = []
    for i in range(NUM_BLOCKS):
        probe_mask = all_enabled.clone()
        probe_mask[i] = 0.0
        ablated_metrics = compute_aznas_scores(model, probe_mask, device, TIN_NUM_CLASSES, ablation_config)
        importance = baseline_metrics["total_loss"] - ablated_metrics["total_loss"]
        importance_scores.append(importance.item())

    sorted_indices = sorted(range(NUM_BLOCKS), key=lambda i: importance_scores[i], reverse=True)

    mask = torch.zeros(NUM_BLOCKS)
    current_flops = 0
    selected_blocks = []

    for idx in sorted_indices:
        if current_flops + flops_list[idx] <= target_flops:
            mask[idx] = 1.0
            current_flops += flops_list[idx]
            selected_blocks.append(idx)

    actual_ratio = current_flops / total_flops

    print(f"Selected blocks: {selected_blocks}")
    print(f"Kept {len(selected_blocks)}/{NUM_BLOCKS} blocks")

    return mask, importance_scores, actual_ratio, selected_blocks

def apply_mask_to_model(model, mask):
    blocks = get_resnet18_blocks(model)
    for i, block in enumerate(blocks):
        if mask[i] == 0:
            block.conv1.weight.data.zero_()
            block.conv2.weight.data.zero_()

def finetune_student_kd(model, teacher, train_loader, val_loader, device, epochs=3):
    print(f"Fine-tuning...")
    optimizer = torch.optim.Adam(model.parameters(), lr=FT_LR)

    for epoch in range(epochs):
        model.train()
        teacher.eval()

        for batch_idx, (images, labels) in enumerate(train_loader):
            if batch_idx >= MAX_BATCHES:
                break

            images, labels = images.to(device), labels.to(device)

            with torch.no_grad():
                teacher_logits = teacher(images)

            student_logits = model(images)
            loss = kd_loss(student_logits, teacher_logits, labels, T=TEMP_KD, alpha=0.5)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    val_acc = evaluate_model(model, val_loader, device)
    print(f"Val Acc: {val_acc:.2f}%")
    return val_acc

print("\nLoading teacher model...")
teacher = load_resnet18_teacher(TIN_NUM_CLASSES, TINYIMAGENET_TEACHER_CKPT, device)
teacher_acc = evaluate_model(teacher, val_loader, device)
print(f"Teacher accuracy: {teacher_acc:.2f}%")

all_results = {}

print("\n" + "="*80)
print("SETUP COMPLETE! Now run each experiment cell below.")
print("="*80)

Using device: cuda
Loading Tiny-ImageNet...

Loading teacher model...
Teacher accuracy: 68.90%

SETUP COMPLETE! Now run each experiment cell below.


In [ ]:
print("\n" + "="*80)
print("EXPERIMENT A: Expressivity Only")
print("="*80)

ablation_config = {
    "use_expressivity": True,
    "use_progressivity": False,
    "use_trainability": False,
    "use_complexity": False,
    "expr_weight": 1.0,
    "prog_weight": 0.0,
    "train_weight": 0.0,
    "comp_weight": 0.0,
}

model_A = load_resnet18_teacher(TIN_NUM_CLASSES, TINYIMAGENET_TEACHER_CKPT, device)
summarizer_A, token_proj_A, encoder_A = train_policy_encoder_ablation(
    model_A, teacher, train_loader, val_loader, device, ablation_config, "A_expressivity_only"
)
mask_A, importance_A, actual_ratio_A, selected_A = materialize_mask_ablation(
    model_A, encoder_A, summarizer_A, token_proj_A, TEST_RATIO, device, ablation_config
)
apply_mask_to_model(model_A, mask_A)
acc_before_A = evaluate_model(model_A, val_loader, device)
print(f"Before FT: {acc_before_A:.2f}%")
acc_after_A = finetune_student_kd(model_A, teacher, train_loader, val_loader, device, FT_EPOCHS)

early_blocks_A = [i for i in selected_A if i < 4]
late_blocks_A = [i for i in selected_A if i >= 4]

all_results["A_expressivity_only"] = {
    "acc_after_ft": acc_after_A,
    "acc_drop": teacher_acc - acc_after_A,
    "selected_blocks": selected_A,
    "num_early": len(early_blocks_A),
    "num_late": len(late_blocks_A),
}

print(f"\nA COMPLETE: {acc_after_A:.2f}%, Early={len(early_blocks_A)}, Late={len(late_blocks_A)}")

del model_A, summarizer_A, token_proj_A, encoder_A
torch.cuda.empty_cache()
gc.collect()




EXPERIMENT A: Expressivity Only

Training: A_expressivity_only
Phase 1: KD Warmup (1 epochs)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


KD Warmup complete
Phase 2: Co-training (2 epochs)
Epoch 1/2 complete
Epoch 2/2 complete

Materializing mask...
Selected blocks: [6, 1, 4, 3]
Kept 4/8 blocks
Before FT: 0.50%
Fine-tuning...
Val Acc: 11.10%

A COMPLETE: 11.10%, Early=2, Late=2


7927

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT B: Progressivity Only")
print("="*80)

ablation_config = {
    "use_expressivity": False,
    "use_progressivity": True,
    "use_trainability": False,
    "use_complexity": False,
    "expr_weight": 0.0,
    "prog_weight": 1.0,
    "train_weight": 0.0,
    "comp_weight": 0.0,
}

model_B = load_resnet18_teacher(TIN_NUM_CLASSES, TINYIMAGENET_TEACHER_CKPT, device)
summarizer_B, token_proj_B, encoder_B = train_policy_encoder_ablation(
    model_B, teacher, train_loader, val_loader, device, ablation_config, "B_progressivity_only"
)
mask_B, importance_B, actual_ratio_B, selected_B = materialize_mask_ablation(
    model_B, encoder_B, summarizer_B, token_proj_B, TEST_RATIO, device, ablation_config
)
apply_mask_to_model(model_B, mask_B)
acc_before_B = evaluate_model(model_B, val_loader, device)
print(f"Before FT: {acc_before_B:.2f}%")
acc_after_B = finetune_student_kd(model_B, teacher, train_loader, val_loader, device, FT_EPOCHS)

early_blocks_B = [i for i in selected_B if i < 4]
late_blocks_B = [i for i in selected_B if i >= 4]

all_results["B_progressivity_only"] = {
    "acc_after_ft": acc_after_B,
    "acc_drop": teacher_acc - acc_after_B,
    "selected_blocks": selected_B,
    "num_early": len(early_blocks_B),
    "num_late": len(late_blocks_B),
}

print(f"\nB COMPLETE: {acc_after_B:.2f}%, Early={len(early_blocks_B)}, Late={len(late_blocks_B)}")

del model_B, summarizer_B, token_proj_B, encoder_B
torch.cuda.empty_cache()
gc.collect()


EXPERIMENT B: Progressivity Only

Training: B_progressivity_only
Phase 1: KD Warmup (1 epochs)
KD Warmup complete
Phase 2: Co-training (2 epochs)
Epoch 1/2 complete
Epoch 2/2 complete

Materializing mask...
Selected blocks: [2, 1, 3, 6]
Kept 4/8 blocks
Before FT: 0.50%
Fine-tuning...
Val Acc: 2.28%

B COMPLETE: 2.28%, Early=3, Late=1


8

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT C: Trainability Only")
print("="*80)

ablation_config = {
    "use_expressivity": False,
    "use_progressivity": False,
    "use_trainability": True,
    "use_complexity": False,
    "expr_weight": 0.0,
    "prog_weight": 0.0,
    "train_weight": 1.0,
    "comp_weight": 0.0,
}

model_C = load_resnet18_teacher(TIN_NUM_CLASSES, TINYIMAGENET_TEACHER_CKPT, device)
summarizer_C, token_proj_C, encoder_C = train_policy_encoder_ablation(
    model_C, teacher, train_loader, val_loader, device, ablation_config, "C_trainability_only"
)
mask_C, importance_C, actual_ratio_C, selected_C = materialize_mask_ablation(
    model_C, encoder_C, summarizer_C, token_proj_C, TEST_RATIO, device, ablation_config
)
apply_mask_to_model(model_C, mask_C)
acc_before_C = evaluate_model(model_C, val_loader, device)
print(f"Before FT: {acc_before_C:.2f}%")
acc_after_C = finetune_student_kd(model_C, teacher, train_loader, val_loader, device, FT_EPOCHS)

early_blocks_C = [i for i in selected_C if i < 4]
late_blocks_C = [i for i in selected_C if i >= 4]

all_results["C_trainability_only"] = {
    "acc_after_ft": acc_after_C,
    "acc_drop": teacher_acc - acc_after_C,
    "selected_blocks": selected_C,
    "num_early": len(early_blocks_C),
    "num_late": len(late_blocks_C),
}

print(f"\nC COMPLETE: {acc_after_C:.2f}%, Early={len(early_blocks_C)}, Late={len(late_blocks_C)}")

del model_C, summarizer_C, token_proj_C, encoder_C
torch.cuda.empty_cache()
gc.collect()


EXPERIMENT C: Trainability Only

Training: C_trainability_only
Phase 1: KD Warmup (1 epochs)
KD Warmup complete
Phase 2: Co-training (2 epochs)
Epoch 1/2 complete
Epoch 2/2 complete

Materializing mask...
Selected blocks: [4, 5, 3, 2]
Kept 4/8 blocks
Before FT: 0.50%
Fine-tuning...
Val Acc: 1.30%

C COMPLETE: 1.30%, Early=2, Late=2


8

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT D: Complexity Only")
print("="*80)

ablation_config = {
    "use_expressivity": False,
    "use_progressivity": False,
    "use_trainability": False,
    "use_complexity": True,
    "expr_weight": 0.0,
    "prog_weight": 0.0,
    "train_weight": 0.0,
    "comp_weight": 1.0,
}

model_D = load_resnet18_teacher(TIN_NUM_CLASSES, TINYIMAGENET_TEACHER_CKPT, device)
summarizer_D, token_proj_D, encoder_D = train_policy_encoder_ablation(
    model_D, teacher, train_loader, val_loader, device, ablation_config, "D_complexity_only"
)
mask_D, importance_D, actual_ratio_D, selected_D = materialize_mask_ablation(
    model_D, encoder_D, summarizer_D, token_proj_D, TEST_RATIO, device, ablation_config
)
apply_mask_to_model(model_D, mask_D)
acc_before_D = evaluate_model(model_D, val_loader, device)
print(f"Before FT: {acc_before_D:.2f}%")
acc_after_D = finetune_student_kd(model_D, teacher, train_loader, val_loader, device, FT_EPOCHS)

early_blocks_D = [i for i in selected_D if i < 4]
late_blocks_D = [i for i in selected_D if i >= 4]

all_results["D_complexity_only"] = {
    "acc_after_ft": acc_after_D,
    "acc_drop": teacher_acc - acc_after_D,
    "selected_blocks": selected_D,
    "num_early": len(early_blocks_D),
    "num_late": len(late_blocks_D),
}

print(f"\nD COMPLETE: {acc_after_D:.2f}%, Early={len(early_blocks_D)}, Late={len(late_blocks_D)}")

del model_D, summarizer_D, token_proj_D, encoder_D
torch.cuda.empty_cache()
gc.collect()


EXPERIMENT D: Complexity Only

Training: D_complexity_only
Phase 1: KD Warmup (1 epochs)
KD Warmup complete
Phase 2: Co-training (2 epochs)
Epoch 1/2 complete
Epoch 2/2 complete

Materializing mask...
Selected blocks: [0, 1, 3]
Kept 3/8 blocks
Before FT: 0.50%
Fine-tuning...
Val Acc: 1.52%

D COMPLETE: 1.52%, Early=3, Late=0


8

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT E: Expr + Prog (Baseline)")
print("="*80)

ablation_config = {
    "use_expressivity": True,
    "use_progressivity": True,
    "use_trainability": False,
    "use_complexity": False,
    "expr_weight": 1.0,
    "prog_weight": 0.1,
    "train_weight": 0.0,
    "comp_weight": 0.0,
}

model_E = load_resnet18_teacher(TIN_NUM_CLASSES, TINYIMAGENET_TEACHER_CKPT, device)
summarizer_E, token_proj_E, encoder_E = train_policy_encoder_ablation(
    model_E, teacher, train_loader, val_loader, device, ablation_config, "E_expr_prog_baseline"
)
mask_E, importance_E, actual_ratio_E, selected_E = materialize_mask_ablation(
    model_E, encoder_E, summarizer_E, token_proj_E, TEST_RATIO, device, ablation_config
)
apply_mask_to_model(model_E, mask_E)
acc_before_E = evaluate_model(model_E, val_loader, device)
print(f"Before FT: {acc_before_E:.2f}%")
acc_after_E = finetune_student_kd(model_E, teacher, train_loader, val_loader, device, FT_EPOCHS)

early_blocks_E = [i for i in selected_E if i < 4]
late_blocks_E = [i for i in selected_E if i >= 4]

all_results["E_expr_prog_baseline"] = {
    "acc_after_ft": acc_after_E,
    "acc_drop": teacher_acc - acc_after_E,
    "selected_blocks": selected_E,
    "num_early": len(early_blocks_E),
    "num_late": len(late_blocks_E),
}

print(f"\nE COMPLETE: {acc_after_E:.2f}%, Early={len(early_blocks_E)}, Late={len(late_blocks_E)}")

del model_E, summarizer_E, token_proj_E, encoder_E
torch.cuda.empty_cache()
gc.collect()


EXPERIMENT E: Expr + Prog (Baseline)

Training: E_expr_prog_baseline
Phase 1: KD Warmup (1 epochs)
KD Warmup complete
Phase 2: Co-training (2 epochs)
Epoch 1/2 complete
Epoch 2/2 complete

Materializing mask...
Selected blocks: [1, 3, 6, 4]
Kept 4/8 blocks
Before FT: 0.50%
Fine-tuning...
Val Acc: 12.62%

E COMPLETE: 12.62%, Early=2, Late=2


8

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT F: All 4 Metrics")
print("="*80)

ablation_config = {
    "use_expressivity": True,
    "use_progressivity": True,
    "use_trainability": True,
    "use_complexity": True,
    "expr_weight": 1.0,
    "prog_weight": 0.1,
    "train_weight": 0.01,
    "comp_weight": 0.01,
}

model_F = load_resnet18_teacher(TIN_NUM_CLASSES, TINYIMAGENET_TEACHER_CKPT, device)
summarizer_F, token_proj_F, encoder_F = train_policy_encoder_ablation(
    model_F, teacher, train_loader, val_loader, device, ablation_config, "F_all_metrics"
)
mask_F, importance_F, actual_ratio_F, selected_F = materialize_mask_ablation(
    model_F, encoder_F, summarizer_F, token_proj_F, TEST_RATIO, device, ablation_config
)
apply_mask_to_model(model_F, mask_F)
acc_before_F = evaluate_model(model_F, val_loader, device)
print(f"Before FT: {acc_before_F:.2f}%")
acc_after_F = finetune_student_kd(model_F, teacher, train_loader, val_loader, device, FT_EPOCHS)

early_blocks_F = [i for i in selected_F if i < 4]
late_blocks_F = [i for i in selected_F if i >= 4]

all_results["F_all_metrics"] = {
    "acc_after_ft": acc_after_F,
    "acc_drop": teacher_acc - acc_after_F,
    "selected_blocks": selected_F,
    "num_early": len(early_blocks_F),
    "num_late": len(late_blocks_F),
}

print(f"\nF COMPLETE: {acc_after_F:.2f}%, Early={len(early_blocks_F)}, Late={len(late_blocks_F)}")

del model_F, summarizer_F, token_proj_F, encoder_F
torch.cuda.empty_cache()
gc.collect()


EXPERIMENT F: All 4 Metrics

Training: F_all_metrics
Phase 1: KD Warmup (1 epochs)
KD Warmup complete
Phase 2: Co-training (2 epochs)
Epoch 1/2 complete
Epoch 2/2 complete

Materializing mask...
Selected blocks: [3, 1, 6, 4]
Kept 4/8 blocks
Before FT: 0.50%
Fine-tuning...
Val Acc: 11.71%

F COMPLETE: 11.71%, Early=2, Late=2


8

In [ ]:
print("\n" + "="*80)
print("ABLATION STUDY COMPLETE!")
print("="*80)

print("\nSUMMARY TABLE")
print("="*100)
print(f"{'Config':<25} {'Acc After FT':<15} {'Acc Drop':<12} {'Early':<10} {'Late':<10} {'Selected Blocks'}")
print("="*100)

for name, result in all_results.items():
    print(f"{name:<25} {result['acc_after_ft']:<15.2f} {result['acc_drop']:<12.2f} "
          f"{result['num_early']:<10} {result['num_late']:<10} {result['selected_blocks']}")

print("="*100)

best_config = max(all_results.items(), key=lambda x: x[1]['acc_after_ft'])
print(f"\nBEST CONFIGURATION: {best_config[0]}")
print(f"Accuracy: {best_config[1]['acc_after_ft']:.2f}%")
print(f"Selected blocks: {best_config[1]['selected_blocks']}")

results_path = os.path.join(OUT_DIR, "aznas_component_ablation_fast_results.json")
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\nResults saved to: {results_path}")



ABLATION STUDY COMPLETE!

SUMMARY TABLE
Config                    Acc After FT    Acc Drop     Early      Late       Selected Blocks
A_expressivity_only       11.10           57.80        2          2          [6, 1, 4, 3]
B_progressivity_only      2.28            66.62        3          1          [2, 1, 3, 6]
C_trainability_only       1.30            67.60        2          2          [4, 5, 3, 2]
D_complexity_only         1.52            67.38        3          0          [0, 1, 3]
E_expr_prog_baseline      12.62           56.28        2          2          [1, 3, 6, 4]
F_all_metrics             11.71           57.19        2          2          [3, 1, 6, 4]

BEST CONFIGURATION: E_expr_prog_baseline
Accuracy: 12.62%
Selected blocks: [1, 3, 6, 4]

Results saved to: checkpoints/aznas_component_ablation_fast_results.json
